In [ ]:
# Bài 17 - Core implementation: Transformer block
# Tự ghép Multi-Head Self-Attention, residual + LayerNorm và FFN bằng PyTorch.
# Không sử dụng nn.TransformerEncoderLayer.

import math
import torch
import torch.nn as nn

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError('d_model phải chia hết cho n_heads')
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attention_dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # x: [B, T, D]; sau tách head: [B, H, T, d_head]
        batch_size, seq_len, _ = x.shape
        def split_heads(tensor):
            return tensor.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        q = split_heads(self.q_proj(x))
        k = split_heads(self.k_proj(x))
        v = split_heads(self.v_proj(x))

        # Attention scores: [B, H, T, T]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if key_padding_mask is not None:
            if key_padding_mask.shape != (batch_size, seq_len):
                raise ValueError('key_padding_mask cần có shape [B, T]')
            # True đánh dấu vị trí padding ở chiều key cần bỏ qua.
            scores = scores.masked_fill(
                key_padding_mask[:, None, None, :].bool(),
                torch.finfo(scores.dtype).min,
            )
        attention_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(self.attention_dropout(attention_weights), v)
        # Nối các head: [B, H, T, d_head] -> [B, T, D]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(context), attention_weights


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.attention_residual_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.ffn_residual_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask=None, return_attention=False):
        attention_output, attention_weights = self.attention(x, key_padding_mask)
        # Add & Norm: LayerNorm(x + Attention(x))
        x = self.norm1(x + self.attention_residual_dropout(attention_output))
        # Add & Norm: LayerNorm(x + FFN(x))
        output = self.norm2(x + self.ffn_residual_dropout(self.ffn(x)))
        if return_attention:
            return output, attention_weights
        return output


# Ví dụ shape theo đề: B=8, T=100, d_model=128, n_heads=4 => d_head=32.
batch_size, seq_len = 8, 100
d_model, n_heads, d_ff = 128, 4, 256
block = TransformerBlock(d_model, n_heads, d_ff, dropout=0.1).to(device)
x = torch.randn(batch_size, seq_len, d_model, device=device)
block.eval()
with torch.no_grad():
    output, attention_weights = block(x, return_attention=True)

print('Input:', tuple(x.shape))
print('Q, K, V sau tách head:', (batch_size, n_heads, seq_len, d_model // n_heads))
print('Attention scores / weights:', tuple(attention_weights.shape))
print('Output:', tuple(output.shape))
print('Số tham số:', sum(p.numel() for p in block.parameters()))

PyTorch: 2.11.0+cpu
Device: cpu
Input: (8, 100, 128)
Q, K, V sau tách head: (8, 4, 100, 32)
Attention scores / weights: (8, 4, 100, 100)
Output: (8, 100, 128)
Số tham số: 132480


In [ ]:
# Bài 18 — Kiểm tra shape thực tế trong quá trình chạy
# Theo dõi tensor từ X qua Q, K, V, Attention, FFN đến output.
block.eval()
with torch.no_grad():
    B, T, D = x.shape
    attention = block.attention
    H = attention.n_heads
    d_head = attention.d_head

    # Q, K, V: [B, T, D] -> [B, H, T, d_head]
    q = attention.q_proj(x).view(B, T, H, d_head).transpose(1, 2)
    k = attention.k_proj(x).view(B, T, H, d_head).transpose(1, 2)
    v = attention.v_proj(x).view(B, T, H, d_head).transpose(1, 2)

    # Attention scores/weights: [B, H, T, d_head] x [B, H, d_head, T] -> [B, H, T, T]
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)
    weights = torch.softmax(scores, dim=-1)
    context = torch.matmul(attention.attention_dropout(weights), v)

    # Nối các head và chiếu về d_model.
    context_concat = context.transpose(1, 2).contiguous().view(B, T, D)
    attention_output = attention.out_proj(context_concat)

    # Residual + LayerNorm, sau đó Feed-Forward Network.
    after_attention = block.norm1(
        x + block.attention_residual_dropout(attention_output)
    )
    ffn_hidden = block.ffn[1](block.ffn[0](after_attention))
    ffn_output = block.ffn[3](block.ffn[2](ffn_hidden))
    output_trace = block.norm2(
        after_attention + block.ffn_residual_dropout(ffn_output)
    )

shape_trace = {
    'X': tuple(x.shape),
    'Q': tuple(q.shape),
    'K': tuple(k.shape),
    'V': tuple(v.shape),
    'Attention scores': tuple(scores.shape),
    'Attention weights': tuple(weights.shape),
    'Attention output (sau nối head + Linear)': tuple(attention_output.shape),
    'FFN hidden (sau Linear đầu + GELU)': tuple(ffn_hidden.shape),
    'FFN output': tuple(ffn_output.shape),
    'Output': tuple(output_trace.shape),
}
for name, shape in shape_trace.items():
    print(f'{name:42s}: {shape}')

# Đối chiếu với cấu hình minh họa của đề Bài 18.
assert shape_trace['X'] == (8, 100, 128)
assert shape_trace['Q'] == shape_trace['K'] == shape_trace['V'] == (8, 4, 100, 32)
assert shape_trace['Attention scores'] == (8, 4, 100, 100)
assert shape_trace['Attention output (sau nối head + Linear)'] == (8, 100, 128)
assert shape_trace['FFN hidden (sau Linear đầu + GELU)'] == (8, 100, 256)
assert shape_trace['FFN output'] == shape_trace['Output'] == (8, 100, 128)
assert torch.allclose(output_trace, output, atol=1e-6), 'Shape trace không khớp forward của block'
print('Tất cả shape đúng; output truy vết khớp với forward của TransformerBlock.')


X                                         : (8, 100, 128)
Q                                         : (8, 4, 100, 32)
K                                         : (8, 4, 100, 32)
V                                         : (8, 4, 100, 32)
Attention scores                          : (8, 4, 100, 100)
Attention weights                         : (8, 4, 100, 100)
Attention output (sau nối head + Linear)  : (8, 100, 128)
FFN hidden (sau Linear đầu + GELU)        : (8, 100, 256)
FFN output                                : (8, 100, 128)
Output                                    : (8, 100, 128)
Tất cả shape đúng; output truy vết khớp với forward của TransformerBlock.
